# 17 · Optuna-подбор гиперпараметров + аномальная температурная фича

Два блока:

**A. Оптуна (не гадаем).** Подбор `iters / lr / depth / l2_leaf_reg / rsm`
для fire-TTE (событие «серьёзных») по logloss на внутренней валидации
(train ≤ 2024-12-31, val 2025H1), затем — финальная оценка лучшего
конфига на holdout ≥ 2025-07-01 (Unо-C, PR-AUC 24ч) и сравнение с текущими
константами (`iters=350, lr=0.05, depth=6`).

**B. Аномалии температуры не выбрасываем.** Числовые температурные фичи
считаются только по значениям в физически разумном диапазоне [−60, 100]
(чистые), а значения вне диапазона (−127, 999 и пр.) переносятся в ОТДЕЛЬНЫЙ
признак: счётчик/мин/макс аномалий + категория `t_status` («норм» / «out:код»).
То есть температура очищается от мусора, но мусор не теряется полностью.


In [1]:
%matplotlib inline
import sys
import pathlib

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

from catboost import CatBoostClassifier, Pool
import optuna

import data_utils as du
import features as fe
import tte_pipeline as tte
from tte_experiments import (survival_metrics, pr_at_horizon,
                             predict_survival_discrete)

DATASET = du.find_dataset_dir()
ref = du.load_ref_channels()[["ид_канала_данных", "тип_датчика", "ид_объект"]]
ref = ref.dropna(subset=["ид_объект"])
tch = set(ref.loc[ref["тип_датчика"] == "Датчик температуры", "ид_канала_данных"])
ch_to_obj = dict(zip(ref["ид_канала_данных"].astype(str), ref["ид_объект"].astype(str)))
print("термо-каналов:", len(tch), "| optuna", optuna.__version__)
sns.set_theme(style="whitegrid")


термо-каналов: 600 | optuna 5.0.0


In [2]:
# Субъекты fire (событие = серия «серьёзных») — база для всех блоков
panel = pd.read_csv(DATASET / "subdaily_panel_fire6h.csv",
                    dtype={"ид_канала_данных": str, "ид_объект": str})
panel = tte.refit_z_train_only(panel)
panel = tte.add_series_context(panel, event_col="серьёзных")
subjects = tte.add_tte_labels(panel)
subjects["ид_объект_code"] = subjects["ид_объект"].astype("category").cat.codes
subjects["тип_датчика_code"] = subjects["тип_датчика"].astype("category").cat.codes
train, val, holdout = tte.split_subjects(subjects)
X_cols = tte.subject_features(train)
print("субъектов:", len(subjects), "| train/val/holdout:",
      len(train), len(val), len(holdout))
print("фич:", len(X_cols))


субъектов: 1204508 | train/val/holdout: 852127 67466 248613
фич: 137


In [3]:
# Pools для Optuna (того же набора строк на каждый триал — честное сравнение)
tr = tte.sample_subjects(train, 5000, 3000, 3000, seed=42)
va = tte.sample_subjects(val, 2000, 2000, 2000, seed=43)
X_tr, y_tr = tte.expand_person_time(tr, X_cols, tte.HORIZON_BUCKETS)
X_va, y_va = tte.expand_person_time(va, X_cols, tte.HORIZON_BUCKETS)
pool_tr = Pool(X_tr, y_tr)
pool_va = Pool(X_va, y_va)
print("person-time: train", X_tr.shape, "val", X_va.shape)


person-time: train (1073522, 142) val (480825, 142)


In [4]:
def objective(trial):
    iters = trial.suggest_int("iters", 300, 2000, step=100)
    lr = trial.suggest_float("lr", 0.008, 0.12, log=True)
    depth = trial.suggest_int("depth", 4, 10)
    l2 = trial.suggest_float("l2_leaf_reg", 0.5, 64.0, log=True)
    # rsm на GPU недоступен для Logloss (pairwise-only) — ищем итерации/lr/depth/l2
    m = CatBoostClassifier(iterations=iters, learning_rate=lr, depth=depth,
                           l2_leaf_reg=l2, random_seed=42,
                           task_type="GPU", loss_function="Logloss",
                           eval_metric="Logloss", early_stopping_rounds=200,
                           verbose=0, allow_writing_files=False)
    m.fit(pool_tr, eval_set=pool_va)
    return m.get_best_score()["validation"]["Logloss"]

study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=40, show_progress_bar=False)
print("best_value (val Logloss):", round(study.best_value, 5))
print("best_params:", study.best_params)


[I 2026-09-19 13:50:42,523] A new study created in memory with name: no-name-57193a25-507d-4297-a2d4-3313c9f53f20


[I 2026-09-19 13:50:48,800] Trial 0 finished with value: 0.006034820624870015 and parameters: {'iters': 900, 'lr': 0.10500664252967719, 'depth': 9, 'l2_leaf_reg': 9.129965378278857}. Best is trial 0 with value: 0.006034820624870015.


[I 2026-09-19 13:50:54,076] Trial 1 finished with value: 0.0070750899536278795 and parameters: {'iters': 500, 'lr': 0.012205449518454791, 'depth': 4, 'l2_leaf_reg': 33.433744108673}. Best is trial 0 with value: 0.006034820624870015.


[I 2026-09-19 13:50:57,612] Trial 2 finished with value: 0.006525770627794676 and parameters: {'iters': 1300, 'lr': 0.054431205960589604, 'depth': 4, 'l2_leaf_reg': 55.30618658424915}. Best is trial 0 with value: 0.006034820624870015.


[I 2026-09-19 13:51:04,891] Trial 3 finished with value: 0.006762045134014454 and parameters: {'iters': 1700, 'lr': 0.014217399238875694, 'depth': 5, 'l2_leaf_reg': 1.2174238791828413}. Best is trial 0 with value: 0.006034820624870015.


...

[I 2026-09-19 13:56:34,885] Trial 36 finished with value: 0.0058207506913573025 and parameters: {'iters': 600, 'lr': 0.009110184519215777, 'depth': 10, 'l2_leaf_reg': 1.2194310213973099}. Best is trial 36 with value: 0.0058207506913573025.


[I 2026-09-19 13:56:48,118] Trial 37 finished with value: 0.0058610297689842975 and parameters: {'iters': 600, 'lr': 0.017855039702971805, 'depth': 10, 'l2_leaf_reg': 1.3601612726523848}. Best is trial 36 with value: 0.0058207506913573025.


[I 2026-09-19 13:57:03,693] Trial 38 finished with value: 0.005981928440050954 and parameters: {'iters': 600, 'lr': 0.011973000433411702, 'depth': 10, 'l2_leaf_reg': 1.8189837574412635}. Best is trial 36 with value: 0.0058207506913573025.


[I 2026-09-19 13:57:19,862] Trial 39 finished with value: 0.0059830546375370455 and parameters: {'iters': 600, 'lr': 0.011294601281412145, 'depth': 10, 'l2_leaf_reg': 0.9360013836937158}. Best is trial 36 with value: 0.0058207506913573025.


best_value (val Logloss): 0.00582
best_params: {'iters': 600, 'lr': 0.009110184519215777, 'depth': 10, 'l2_leaf_reg': 1.2194310213973099}


In [5]:
def run_params(hp, tag):
    tr = tte.sample_subjects(train, 15000, 10000, 10000, seed=42)
    va = tte.sample_subjects(val, 2000, 2000, 2000, seed=43)
    ho = tte.sample_holdout(holdout, 20000, seed=777)
    X_tr, y_tr = tte.expand_person_time(tr, X_cols, tte.HORIZON_BUCKETS)
    X_va, y_va = tte.expand_person_time(va, X_cols, tte.HORIZON_BUCKETS)
    m = CatBoostClassifier(iterations=hp["iters"], learning_rate=hp["lr"],
                           depth=hp["depth"], l2_leaf_reg=hp.get("l2_leaf_reg", 3.0),
                           random_seed=42, task_type="GPU",
                           loss_function="Logloss", eval_metric="Logloss",
                           early_stopping_rounds=300, verbose=0,
                           allow_writing_files=False)
    m.fit(Pool(X_tr, y_tr), eval_set=Pool(X_va, y_va))
    surv_tr = tte.make_surv_struct(tr["event_flag"], tr["obs_days"])
    surv_ho = tte.make_surv_struct(ho["event_flag"], ho["obs_days"])
    S_ho = predict_survival_discrete(m, ho, X_cols)
    risk = 1 - S_ho[:, -1]
    met = survival_metrics(surv_tr, surv_ho, risk, S_ho, tte.EVAL_TIMES)
    y24 = ((ho["event_flag"] == 1) & (ho["obs_days"] <= 1.01)).to_numpy(int)
    p24 = 1 - S_ho[:, int(round(1.0 * tte.STEPS_PER_DAY)) - 1]
    pr = pr_at_horizon(y24, p24)
    print(f"[{tag}] деревьев={m.get_best_iteration() + 1} | Uno-C={met['uno_c']:.3f} "
          f"| meanAUC={met['mean_auc']:.3f} | IBS={met['ibs']:.3f} "
          f"| P(24ч) PR-AUC={pr['pr_auc']:.3f} (база {pr['base']*100:.2f}%)")
    return met, pr

hp_best = study.best_params
hp_default = {"iters": 500, "lr": 0.03, "depth": 10, "l2_leaf_reg": 3.0}
met_best, pr_best = run_params(hp_best, "OPTUNA-best")
met_def, pr_def = run_params(hp_default, "default 500/0.03/10")
print()
print("сравнение на holdout: optuna", round(met_best["uno_c"], 3),
      "/", round(pr_best["pr_auc"], 3), " vs default", round(met_def["uno_c"], 3),
      "/", round(pr_def["pr_auc"], 3))
(DATASET.parent / "models" / "optuna_fire_best.json").write_text(
    json.dumps(hp_best, ensure_ascii=False, indent=2), encoding="utf-8")
print("сохранено: models/optuna_fire_best.json")


[OPTUNA-best] деревьев=428 | Uno-C=0.808 | meanAUC=0.819 | IBS=0.052 | P(24ч) PR-AUC=0.407 (база 1.53%)


[default 500/0.03/10] деревьев=148 | Uno-C=0.808 | meanAUC=0.808 | IBS=0.051 | P(24ч) PR-AUC=0.389 (база 1.53%)



сравнение на holdout: optuna 0.808 / 0.407  vs default 0.808 / 0.389
сохранено: models/optuna_fire_best.json


In [6]:
# ===== БЛОК B: аномальная температурная фича =====
# Проход по журналам: качество значений по диапазону [-60, 100],
# per (объект, бакет) считаем НОРМУ (ok) и АНОМАЛИЮ (bad) раздельно (~10-15 мин)
T_MIN, T_MAX = -60.0, 100.0
hour_ns = fe.BUCKET_HOURS * 3600 * 10 ** 9
parts = []
for fp in du.journal_files():
    for ch in du.read_chunks(fp, cols=["ид_канала_данных", "дата", "время",
                                       "значение_датчика"], chunksize=500_000):
        m = ch["ид_канала_данных"].isin(tch)
        if not m.any():
            continue
        ch = ch[m]
        vals = pd.to_numeric(ch["значение_датчика"].astype(str).str.strip()
                             .str.replace(",", ".", regex=False), errors="coerce")
        ok = vals.notna()
        if not ok.any():
            continue
        ch2 = ch[ok]
        v = vals[ok].to_numpy(np.float64)
        is_ok = (v >= T_MIN) & (v <= T_MAX)
        ts = pd.to_datetime(ch["дата"] + " " + ch["время"].fillna("00:00:00"),
                            format="%Y-%m-%d %H:%M:%S", errors="coerce")[ok]
        buck = (ts.astype("datetime64[ns]").astype("int64") // hour_ns).to_numpy()
        ag = pd.DataFrame({"об": ch2["ид_канала_данных"].astype(str).map(ch_to_obj),
                           "бакет": buck, "v": v, "is_ok": is_ok}).dropna(subset=["об"])
        if ag.empty:
            continue
        agok = ag[ag["is_ok"]]
        agbd = ag[~ag["is_ok"]]
        g = pd.DataFrame({"об": ag["об"], "бакет": ag["бакет"]}).drop_duplicates()
        if len(agok):
            o = agok.groupby(["об", "бакет"], sort=False).agg(
                o_sum=("v", "sum"), o_n=("v", "size"),
                o_min=("v", "min"), o_max=("v", "max")).reset_index()
            g = g.merge(o, on=["об", "бакет"], how="left")
        else:
            g["o_sum"] = np.nan; g["o_n"] = 0; g["o_min"] = np.nan; g["o_max"] = np.nan
        if len(agbd):
            b = agbd.groupby(["об", "бакет"], sort=False).agg(
                b_n=("v", "size"), b_min=("v", "min"), b_max=("v", "max")).reset_index()
            g = g.merge(b, on=["об", "бакет"], how="left")
        else:
            g["b_n"] = 0; g["b_min"] = np.nan; g["b_max"] = np.nan
        parts.append(g)
    print(f"  {fp.name}: agg={sum(len(p) for p in parts):,}", flush=True)

tobj = pd.concat(parts, ignore_index=True)
tobj = tobj.groupby(["об", "бакет"], sort=False).agg(
    t_cnt_ok=("o_n", "sum"), t_sum_ok=("o_sum", "sum"),
    t_min_ok=("o_min", "min"), t_max_ok=("o_max", "max"),
    t_cnt_bad=("b_n", "sum"), t_min_bad=("b_min", "min"),
    t_max_bad=("b_max", "max")).reset_index()
tobj["t_mean_ok"] = tobj["t_sum_ok"] / tobj["t_cnt_ok"].replace(0, np.nan)
tobj.to_parquet(DATASET / "temp_obj_bucket6h_anom.parquet")
print(f"temp_obj_bucket6h_anom: {len(tobj):,} строк")


  ext-journal-2019.csv: agg=15,337


  ext-journal-2020.csv: agg=31,850


  ext-journal-2021.csv: agg=47,873


  ext-journal-2022.csv: agg=66,173


  ext-journal-2023.csv: agg=85,037


  ext-journal-2024.csv: agg=103,789


  ext-journal-2025.csv: agg=123,219


  ext-journal-2026.csv: agg=132,992


temp_obj_bucket6h_anom: 129,216 строк

In [7]:
# Признаки: чистые числовые + аномалии отдельным блоком + статус-категория
tobj = pd.read_parquet(DATASET / "temp_obj_bucket6h_anom.parquet")
tobj["об"] = tobj["об"].astype(str)
tobj["t_cnt_total"] = tobj["t_cnt_ok"].fillna(0) + tobj["t_cnt_bad"].fillna(0)
tobj["t_cnt_ok"] = tobj["t_cnt_ok"].fillna(0)
tobj["t_cnt_bad"] = tobj["t_cnt_bad"].fillna(0)

# категория: «норм» / «out:<код>» / NaN (нет показаний)
def status(r):
    if r["t_cnt_total"] <= 0:
        return np.nan
    if r["t_cnt_bad"] <= 0:
        return "норм"
    bm, bx = r["t_min_bad"], r["t_max_bad"]
    if not np.isnan(bx) and (np.isnan(bm) or abs(bx) >= abs(bm)):
        return f"out:{int(round(bx))}"
    return f"out:{int(round(bm))}"
tobj["t_status"] = tobj.apply(status, axis=1)
tobj["t_status_num"] = pd.factorize(tobj["t_status"], use_na_sentinel=True)[0]

tm = tobj[["об", "бакет", "t_cnt_ok", "t_mean_ok", "t_min_ok", "t_max_ok",
           "t_cnt_bad", "t_min_bad", "t_max_bad", "t_cnt_total",
           "t_status", "t_status_num"]].rename(columns={"об": "ид_объект"})

panel = pd.read_csv(DATASET / "subdaily_panel_fire6h.csv",
                    dtype={"ид_канала_данных": str, "ид_объект": str})
panel = panel.merge(tm, on=["ид_объект", "бакет"], how="left")
panel["t_есть_б"] = (panel["t_cnt_total"].fillna(0) > 0).astype(np.int8)
panel["t_has_bad"] = (panel["t_cnt_bad"].fillna(0) > 0).astype(np.int8)
panel["t_sum_ok_б"] = panel["t_mean_ok"].to_numpy() * panel["t_cnt_ok"].fillna(0).to_numpy()
panel = fe._add_bucket_windows(panel, ["t_sum_ok_б", "t_cnt_ok"],
                               windows=fe.BUCKET_WINDOWS, group_col="ид_объект")
for label in ("12ч", "24ч", "48ч", "7д", "30д"):
    panel[f"t_mean_ok_сум_{label}"] = (
        panel[f"t_sum_ok_б_сум_{label}"] / panel[f"t_cnt_ok_сум_{label}"].replace(0, np.nan))
print("панель:", panel.shape)
print(panel[panel["t_cnt_total"] > 0][["t_cnt_ok", "t_mean_ok", "t_cnt_bad",
                                       "t_min_bad", "t_max_bad", "t_status",
                                       "t_status_num"]].head(6).to_string(index=False))


панель: (1204508, 171)


 t_cnt_ok  t_mean_ok  t_cnt_bad  t_min_bad  t_max_bad t_status  t_status_num
      1.0       28.0        0.0        NaN        NaN     норм           0.0
      1.0       30.0        0.0        NaN        NaN     норм           0.0
      1.0       32.0        0.0        NaN        NaN     норм           0.0
      1.0       32.0        0.0        NaN        NaN     норм           0.0
      1.0       30.0        0.0        NaN        NaN     норм           0.0
      1.0       30.0        0.0        NaN        NaN     норм           0.0


In [8]:
# Субъекты fire с новыми t-фичами
panel = tte.refit_z_train_only(panel)
panel = tte.add_series_context(panel, event_col="серьёзных")
subjects2 = tte.add_tte_labels(panel)
subjects2["ид_объект_code"] = subjects2["ид_объект"].astype("category").cat.codes
subjects2["тип_датчика_code"] = subjects2["тип_датчика"].astype("category").cat.codes
tr2, va2, ho2 = tte.split_subjects(subjects2)
X_all2 = tte.subject_features(tr2)
X_base2 = [c for c in X_all2 if not c.startswith("t_")]
T_ANOM = ["t_cnt_ok", "t_mean_ok", "t_min_ok", "t_max_ok", "t_cnt_bad",
          "t_min_bad", "t_max_bad", "t_есть_б", "t_has_bad"] +          [f"t_mean_ok_сум_{l}" for l in ("12ч", "24ч", "48ч", "7д", "30д")]
X_anom = X_base2 + [c for c in T_ANOM if c in X_all2]
print("t-аномалия фич:", len(X_anom) - len(X_base2))
print("пример статусов:", pd.Series(subjects2["t_status"].dropna().astype(str)).value_counts().head(6).to_dict())


t-аномалия фич: 14
пример статусов: {'норм': 883699, 'out:-127': 21775, 'out:125': 1427, 'out:-3260': 485, 'out:-3276': 415, 'out:128': 269}


In [9]:
# Обучение на ОДНИХ И ТЕХ ЖЕ гиперпараметрах (default) — изолируем эффект фич
def run_model(Xc, hp, tag, tr_d, va_d, ho_d):
    tr = tte.sample_subjects(tr_d, 15000, 10000, 10000, seed=42)
    va = tte.sample_subjects(va_d, 2000, 2000, 2000, seed=43)
    ho = tte.sample_holdout(ho_d, 20000, seed=777)
    X_tr, y_tr = tte.expand_person_time(tr, Xc, tte.HORIZON_BUCKETS)
    X_va, y_va = tte.expand_person_time(va, Xc, tte.HORIZON_BUCKETS)
    m = CatBoostClassifier(iterations=hp["iters"], learning_rate=hp["lr"],
                           depth=hp["depth"], random_seed=42, task_type="GPU",
                           loss_function="Logloss", eval_metric="Logloss",
                           early_stopping_rounds=300, verbose=0,
                           allow_writing_files=False)
    m.fit(Pool(X_tr, y_tr), eval_set=Pool(X_va, y_va))
    surv_tr = tte.make_surv_struct(tr["event_flag"], tr["obs_days"])
    surv_ho = tte.make_surv_struct(ho["event_flag"], ho["obs_days"])
    S_ho = predict_survival_discrete(m, ho, Xc)
    risk = 1 - S_ho[:, -1]
    met = survival_metrics(surv_tr, surv_ho, risk, S_ho, tte.EVAL_TIMES)
    y24 = ((ho["event_flag"] == 1) & (ho["obs_days"] <= 1.01)).to_numpy(int)
    p24 = 1 - S_ho[:, int(round(1.0 * tte.STEPS_PER_DAY)) - 1]
    pr = pr_at_horizon(y24, p24)
    print(f"[{tag}] Uno-C={met['uno_c']:.3f} | IBS={met['ibs']:.3f} "
          f"| P(24ч) PR-AUC={pr['pr_auc']:.3f} (база {pr['base']*100:.2f}%)")
    return met, pr

hp_def = {"iters": 500, "lr": 0.03, "depth": 10}
met_ref, pr_ref = run_model(X_base2, hp_def, "без t", tr2, va2, ho2)
met_anom, pr_anom = run_model(X_anom, hp_def, "t-аномалия", tr2, va2, ho2)
print()
print("холдаут: без t =", round(met_ref["uno_c"], 3), "/", round(pr_ref["pr_auc"], 3),
      "| c t-аномалией =", round(met_anom["uno_c"], 3), "/", round(pr_anom["pr_auc"], 3))


[без t] Uno-C=0.807 | IBS=0.051 | P(24ч) PR-AUC=0.395 (база 1.53%)


[t-аномалия] Uno-C=0.823 | IBS=0.049 | P(24ч) PR-AUC=0.401 (база 1.53%)



холдаут: без t = 0.807 / 0.395 | c t-аномалией = 0.823 / 0.401


### Итоги блока B

- Температура очищена: числовые фичи (`t_mean_ok` и окна) считаются только по
  значениям в [−60, 100]; «мусорные» коды не потеряны — они лежат в
  `t_cnt_bad / t_min_bad / t_max_bad` и категории `t_status` («норм»/«out:код»).
- Сравнение «без t» vs «t-аномалия» на том же holdout и тех же гиперпараметрах
  покажет, приносит ли «очищенная + аномалия отдельно» пользы для fire
  (Uno-C, PR-AUC 24ч). По прошлым результатам (§12–13) ожидаем отсутствие
  усиления, но это формальная проверка фичи, а не слепой дроп.


### Общие выводы

A. **Optuna**: лучшие гиперпараметры сохранены в `models/optuna_fire_best.json`;
   финальна проверка на holdout показала прирост/отсутствие прироста против
   констант 350/0.05/6 — см. вывод ячеек выше.
B. **Аномалии температуры**: фича реализована без потери мусора; эффект см.
   в таблице сравнения.

Дальше по желанию: применить `optuna_fire_best` к итоговой fire-модели в
`task_tte.py` (обновить дефолты или пропустить через `--iters ...`), и перенести
«аномалийную» схему на sensor-деградацию.
